# P10.6-AI — Notebook 63: entrenamiento subarticular Axial T2

Entrena un clasificador **2.5D multiclase** para estenosis subarticular izquierda y derecha usando exclusivamente `train_manifest.csv` y `validation_manifest.csv` del Notebook 62.

El backbone se comparte entre ambos lados e incorpora embeddings explícitos de **lado** y **nivel lumbar**. El `internal_test` permanece sellado para el Notebook 64.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Recursos y alcance

- Runtime recomendado: **Google Colab con GPU T4 o superior**.
- Autorizar Google Drive.
- No requiere token de GitHub.
- El token de Kaggle se solicita únicamente si las series Axial T2 necesarias no están disponibles en Drive.
- El caché local de crops `.npy` puede reutilizarse mientras no se reinicie el runtime.
- No abre ni carga `internal_test_manifest.csv`; solo verifica su existencia y el sello criptográfico.


In [1]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util, subprocess, sys

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing
    ])
print({"installedNow": missing})


{'installedNow': ['pydicom']}


In [2]:
# 2) Runtime y Google Drive — compatible con CPU o GPU

import torch
from google.colab import drive

runtime_device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

runtime_info = {
    "device": runtime_device,
    "torch": torch.__version__,
}

if torch.cuda.is_available():
    runtime_info.update({
        "gpu": torch.cuda.get_device_name(0),
        "gpuMemoryGiB": round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            2,
        ),
    })

print(runtime_info)

drive.mount(
    "/content/drive",
    force_remount=False,
)

{'device': 'cpu', 'torch': '2.11.0+cpu'}
Mounted at /content/drive


In [4]:
# 3) Clonar o actualizar la rama e importar el pipeline

from pathlib import Path
import subprocess
import sys

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)

REPO_REF = "enzo/p10-6-ai-rsna-findings"

REPO_ROOT = Path(
    "/content/PFI_MVPTest_Enzo_AImodule"
)

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        "--branch",
        REPO_REF,
        "--single-branch",
        REPO_URL,
        str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        [
            "git",
            "fetch",
            "origin",
            REPO_REF,
        ],
        cwd=REPO_ROOT,
    )

    subprocess.check_call(
        [
            "git",
            "checkout",
            REPO_REF,
        ],
        cwd=REPO_ROOT,
    )

    subprocess.check_call(
        [
            "git",
            "pull",
            "--ff-only",
            "origin",
            REPO_REF,
        ],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO_ROOT,
    text=True,
).strip()

ai_service_path = str(
    REPO_ROOT / "ai_service"
)

if ai_service_path not in sys.path:
    sys.path.insert(
        0,
        ai_service_path,
    )

from pfi_ai_service.training.rsna_subarticular_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({
    "repoRef": REPO_REF,
    "repoSha": REPO_SHA,
    "repoRoot": str(REPO_ROOT),
})

{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': '22cda3e1dfcf6aaa110c44d1993371d08017edba', 'repoRoot': '/content/PFI_MVPTest_Enzo_AImodule'}


In [5]:
# 4) Rutas y configuración — caché persistente en Drive

from pathlib import Path

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")

RESULTS_ROOT = (
    PFI_ROOT
    / "results"
    / "P10_6_rsna_findings"
)

SPLIT_ROOT = (
    RESULTS_ROOT
    / "notebook62_subarticular_split"
)

RUN_ROOT = (
    RESULTS_ROOT
    / "notebook63_subarticular_training"
)

MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
    / "subarticular_axial_t2_2p5d"
)

CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path(
    "/content/RSNA_LUMBAR_DISC"
)

DRIVE_DATA_ROOT = (
    PFI_ROOT
    / "data"
    / "RSNA_LUMBAR_DISC"
)

# IMPORTANTE:
# El caché queda persistido en Drive, no en /content.
CACHE_ROOT = (
    PFI_ROOT
    / "cache"
    / "notebook63_subarticular_cache"
)

COMPETITION = (
    "rsna-2024-lumbar-spine-degenerative-classification"
)

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
    minimum_macro_f1=0.36,
    minimum_balanced_accuracy=0.45,
    minimum_severe_recall=0.30,
    minimum_moderate_recall=0.25,
)

for path in (
    RUN_ROOT,
    MODEL_ROOT,
    CHECKPOINT_ROOT,
    CACHE_ROOT,
):
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

print(CFG)

print({
    "splitRoot": str(SPLIT_ROOT),
    "runRoot": str(RUN_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
    "cacheRoot": str(CACHE_ROOT),
    "cachePersistent": True,
    "cacheExists": CACHE_ROOT.exists(),
})

TrainConfig(seed=2026, image_size=224, crop_size=256, batch_size=32, num_workers=2, max_epochs=15, patience=5, learning_rate=0.0002, weight_decay=0.0001, model_name='efficientnet_b0', pretrained=True, side_embedding_dim=8, level_embedding_dim=12, dropout=0.25, label_smoothing=0.05, severe_loss_multiplier=1.25, max_grad_norm=2.0, minimum_macro_f1=0.36, minimum_balanced_accuracy=0.45, minimum_severe_recall=0.3, minimum_moderate_recall=0.25)
{'splitRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook62_subarticular_split', 'runRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook63_subarticular_training', 'checkpointRoot': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/subarticular_axial_t2_2p5d/checkpoints', 'cacheRoot': '/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache', 'cachePersistent': True, 'cacheExists': True}


## Carga y auditoría de datos

La siguiente celda verifica hashes, aprobación del Notebook 62, separación por `study_id`, presencia de las tres clases y el sello del internal test. Solo lee `train` y `validation`.


In [6]:
# 5) Cargar únicamente train y validation
train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

train_distribution = (
    train_manifest
    .groupby(["side", "level", "severity"])
    .size()
    .reset_index(name="train_rows")
)
validation_distribution = (
    validation_manifest
    .groupby(["side", "level", "severity"])
    .size()
    .reset_index(name="validation_rows")
)

print({
    "trainRows": len(train_manifest),
    "trainStudies": train_manifest["study_id"].nunique(),
    "validationRows": len(validation_manifest),
    "validationStudies": validation_manifest["study_id"].nunique(),
    "trainClassCounts": train_manifest["severity"].value_counts().to_dict(),
    "validationClassCounts": (
        validation_manifest["severity"].value_counts().to_dict()
    ),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
})
display(train_distribution)
display(validation_distribution)


{'trainRows': 13445, 'trainStudies': 1382, 'validationRows': 2894, 'validationStudies': 296, 'trainClassCounts': {'normal_mild': 9615, 'moderate': 2534, 'severe': 1296}, 'validationClassCounts': {'normal_mild': 2052, 'moderate': 562, 'severe': 280}, 'internalTestAccessed': False, 'officialTestAccessed': False}


,side,level,severity,train_rows
0,left,L1-L2,moderate,69
1,left,L1-L2,normal_mild,1176
2,left,L1-L2,severe,18
3,left,L2-L3,moderate,178
4,left,L2-L3,normal_mild,1079
5,left,L2-L3,severe,63
6,left,L3-L4,moderate,315
7,left,L3-L4,normal_mild,929
8,left,L3-L4,severe,135
9,left,L4-L5,moderate,422


,side,level,severity,validation_rows
0,left,L1-L2,moderate,13
1,left,L1-L2,normal_mild,259
2,left,L1-L2,severe,5
3,left,L2-L3,moderate,34
4,left,L2-L3,normal_mild,240
5,left,L2-L3,severe,13
6,left,L3-L4,moderate,59
7,left,L3-L4,normal_mild,204
8,left,L3-L4,severe,32
9,left,L4-L5,moderate,107


In [7]:
# 6) Resolver las series DICOM necesarias con progreso

from pathlib import Path
import time

import pandas as pd
from tqdm.auto import tqdm

from pfi_ai_service.training.rsna_subarticular_training import (
    DataRootAudit,
)


def required_series_table_with_progress(
    *frames: pd.DataFrame,
) -> pd.DataFrame:
    """Obtiene las combinaciones únicas estudio-serie requeridas."""
    combined = pd.concat(
        [
            frame[[
                "study_id",
                "coordinate_series_id",
            ]]
            for frame in frames
        ],
        ignore_index=True,
    )

    combined["study_id"] = (
        combined["study_id"].astype(str)
    )
    combined["coordinate_series_id"] = (
        combined["coordinate_series_id"].astype(str)
    )

    return (
        combined
        .drop_duplicates()
        .sort_values([
            "study_id",
            "coordinate_series_id",
        ])
        .reset_index(drop=True)
    )


def audit_data_root_with_progress(
    root: Path,
    *frames: pd.DataFrame,
) -> DataRootAudit:
    """
    Comprueba que cada serie requerida tenga al menos
    un archivo DICOM y muestra progreso.
    """
    root = Path(root)
    required = required_series_table_with_progress(
        *frames
    )

    train_images_root = root / "train_images"

    print({
        "auditRoot": str(root),
        "requiredSeries": len(required),
        "trainImagesExists": train_images_root.is_dir(),
    })

    # Evita recorrer miles de filas cuando el root ni siquiera
    # contiene la carpeta train_images.
    if not train_images_root.is_dir():
        missing = [
            (
                f"{row.study_id}/"
                f"{row.coordinate_series_id}"
            )
            for row in required.itertuples(index=False)
        ]

        return DataRootAudit(
            root=str(root),
            complete=False,
            required_series=int(len(required)),
            missing_series=int(len(missing)),
            missing_examples=tuple(missing[:20]),
        )

    missing = []
    started = time.time()

    progress = tqdm(
        required.itertuples(index=False),
        total=len(required),
        desc=f"Auditando {root.name}",
        unit="serie",
        mininterval=0.5,
    )

    for index, row in enumerate(progress, start=1):
        series_path = (
            train_images_root
            / str(row.study_id)
            / str(row.coordinate_series_id)
        )

        has_dicom = (
            series_path.is_dir()
            and next(
                series_path.glob("*.dcm"),
                None,
            )
            is not None
        )

        if not has_dicom:
            missing.append(
                (
                    f"{row.study_id}/"
                    f"{row.coordinate_series_id}"
                )
            )

        if index % 100 == 0 or index == len(required):
            progress.set_postfix({
                "faltantes": len(missing),
                "min": round(
                    (time.time() - started) / 60,
                    1,
                ),
            })

    return DataRootAudit(
        root=str(root),
        complete=len(missing) == 0,
        required_series=int(len(required)),
        missing_series=int(len(missing)),
        missing_examples=tuple(missing[:20]),
    )


data_root = None
data_audits = []

for candidate in [
    LOCAL_DATA_ROOT,
    DRIVE_DATA_ROOT,
]:
    print()
    print(f"Revisando: {candidate}")

    audit = audit_data_root_with_progress(
        candidate,
        train_manifest,
        validation_manifest,
    )
    data_audits.append(audit)

    print({
        "root": audit.root,
        "complete": audit.complete,
        "requiredSeries": audit.required_series,
        "missingSeries": audit.missing_series,
        "missingExamples": list(
            audit.missing_examples
        ),
    })

    if audit.complete:
        data_root = candidate
        break


if data_root is None:
    print(
        "No se encontró un root completo. "
        "Se solicitará el token de Kaggle sin mostrarlo."
    )

    kaggle_token = getpass.getpass(
        "Kaggle API token: "
    )

    data_root = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        kaggle_token,
    )


print({
    "selectedDataRoot": str(data_root),
    "dataAudits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "requiredSeries": (
                audit.required_series
            ),
            "missingSeries": (
                audit.missing_series
            ),
        }
        for audit in data_audits
    ],
})


Revisando: /content/RSNA_LUMBAR_DISC
{'auditRoot': '/content/RSNA_LUMBAR_DISC', 'requiredSeries': 1990, 'trainImagesExists': False}
{'root': '/content/RSNA_LUMBAR_DISC', 'complete': False, 'requiredSeries': 1990, 'missingSeries': 1990, 'missingExamples': ['100206310/1012284084', '1002894806/1252873726', '1004726367/992525108', '1008446160/3775545364', '1009445512/1705522953', '1009445512/4018190332', '1012375618/588002243', '1013589491/598943280', '1013791258/821987258', '1018005303/1049505285', '1018005303/3675524442', '1018005303/4193900495', '1019430579/4056780644', '1020394063/3995675145', '1025265129/3119430323', '1028909382/1199603355', '1028909382/1779061941', '1035170868/2484927966', '1036203708/2773479263', '1038453736/3941342785']}

Revisando: /content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC
{'auditRoot': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'requiredSeries': 1990, 'trainImagesExists': True}


Auditando RSNA_LUMBAR_DISC:   0%|          | 0/1990 [00:00<?, ?serie/s]

{'root': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'complete': True, 'requiredSeries': 1990, 'missingSeries': 0, 'missingExamples': []}
{'selectedDataRoot': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'dataAudits': [{'root': '/content/RSNA_LUMBAR_DISC', 'complete': False, 'requiredSeries': 1990, 'missingSeries': 1990}, {'root': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'complete': True, 'requiredSeries': 1990, 'missingSeries': 0}]}


In [8]:
# Auditar el caché persistente antes de continuar

from pathlib import Path
import json

expected_cache = (
    Path("/content/drive/MyDrive/PFI_MVP")
    / "cache"
    / "notebook63_subarticular_cache"
)

CACHE_ROOT = expected_cache

train_root = CACHE_ROOT / "train"
validation_root = CACHE_ROOT / "validation"

# Eliminar únicamente temporales incompletos.
temporary_files = list(
    CACHE_ROOT.rglob("*.tmp.npy")
)

for file in temporary_files:
    file.unlink(missing_ok=True)

train_files = list(
    train_root.glob("*.npy")
)

validation_files = list(
    validation_root.glob("*.npy")
)

status = {
    "cacheRoot": str(CACHE_ROOT),
    "persistent": str(CACHE_ROOT).startswith(
        "/content/drive/"
    ),
    "trainFiles": len(train_files),
    "validationFiles": len(validation_files),
    "temporaryFilesRemoved": len(temporary_files),
    "totalCacheFiles": (
        len(train_files)
        + len(validation_files)
    ),
}

print(json.dumps(
    status,
    indent=2,
    ensure_ascii=False,
))

if not status["persistent"]:
    raise RuntimeError(
        "CACHE_ROOT no está dentro de Drive."
    )

{
  "cacheRoot": "/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache",
  "persistent": true,
  "trainFiles": 3687,
  "validationFiles": 0,
  "temporaryFilesRemoved": 0,
  "totalCacheFiles": 3687
}


In [9]:
from pathlib import Path

print({
    "PFI_ROOT": str(PFI_ROOT),
    "CACHE_ROOT": str(CACHE_ROOT),
    "cacheExists": CACHE_ROOT.exists(),
    "persistent": str(CACHE_ROOT).startswith("/content/drive/"),
    "trainCached": len(list((CACHE_ROOT / "train").glob("*.npy"))),
    "validationCached": len(list((CACHE_ROOT / "validation").glob("*.npy"))),
})

assert str(CACHE_ROOT).startswith("/content/drive/"), (
    "No ejecutes la celda 7: el caché sigue siendo temporal."
)

{'PFI_ROOT': '/content/drive/MyDrive/PFI_MVP', 'CACHE_ROOT': '/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache', 'cacheExists': True, 'persistent': True, 'trainCached': 3687, 'validationCached': 0}


In [10]:
# 7) Preparar muestras y construir caché 2.5D
train_samples = prepare_samples(
    train_manifest,
    data_root,
    "train",
)
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

train_cache_audit = build_cache(
    train_samples,
    CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    CACHE_ROOT,
    "validation",
    CFG,
)

print({
    "trainCache": train_cache_audit,
    "validationCache": validation_cache_audit,
    "internalTestAccessed": False,
})


cache train por serie:   0%|          | 0/1631 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
# Guardar y auditar el progreso antes de cerrar

from pathlib import Path
import json
import time

# Dar unos segundos a Drive para terminar escrituras pendientes.
time.sleep(5)

if not str(CACHE_ROOT).startswith("/content/drive/"):
    raise RuntimeError(
        f"El caché no es persistente: {CACHE_ROOT}"
    )

temporary_files = list(
    CACHE_ROOT.rglob("*.tmp.npy")
)

# La celda 7 ya debe estar completamente detenida.
for file in temporary_files:
    file.unlink(missing_ok=True)

train_files = list(
    (CACHE_ROOT / "train").glob("*.npy")
)

validation_files = list(
    (CACHE_ROOT / "validation").glob("*.npy")
)

report = {
    "cacheRoot": str(CACHE_ROOT),
    "persistent": True,
    "trainCached": len(train_files),
    "validationCached": len(validation_files),
    "temporaryFilesRemoved": len(temporary_files),
    "safeToClose": True,
}

print(json.dumps(
    report,
    indent=2,
    ensure_ascii=False,
))

{
  "cacheRoot": "/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache",
  "persistent": true,
  "trainCached": 9457,
  "validationCached": 0,
  "temporaryFilesRemoved": 0,
  "safeToClose": true
}


In [ ]:
from pathlib import Path
import json

print(json.dumps({
    "cacheRoot": str(CACHE_ROOT),
    "persistent": str(CACHE_ROOT).startswith("/content/drive/"),
    "trainCached": len(
        list((CACHE_ROOT / "train").glob("*.npy"))
    ),
    "validationCached": len(
        list((CACHE_ROOT / "validation").glob("*.npy"))
    ),
}, indent=2))

In [ ]:
# Verificar progreso persistido

from pathlib import Path
import json

train_expected = len(train_samples)
validation_expected = len(validation_samples)

train_present = len(list(
    (CACHE_ROOT / "train").glob("*.npy")
))

validation_present = len(list(
    (CACHE_ROOT / "validation").glob("*.npy")
))

train_percent = (
    100.0 * train_present / train_expected
    if train_expected
    else 0.0
)

validation_percent = (
    100.0
    * validation_present
    / validation_expected
    if validation_expected
    else 0.0
)

report = {
    "cacheRoot": str(CACHE_ROOT),
    "train": {
        "present": train_present,
        "expected": train_expected,
        "percent": round(train_percent, 2),
    },
    "validation": {
        "present": validation_present,
        "expected": validation_expected,
        "percent": round(validation_percent, 2),
    },
    "complete": (
        train_present == train_expected
        and validation_present
        == validation_expected
    ),
}

print(json.dumps(
    report,
    indent=2,
    ensure_ascii=False,
))

In [ ]:
from pathlib import Path
import json

train_expected = len(train_samples)
validation_expected = len(validation_samples)

train_present = len(
    list((CACHE_ROOT / "train").glob("*.npy"))
)

validation_present = len(
    list((CACHE_ROOT / "validation").glob("*.npy"))
)

report = {
    "cacheRoot": str(CACHE_ROOT),
    "train": {
        "present": train_present,
        "expected": train_expected,
    },
    "validation": {
        "present": validation_present,
        "expected": validation_expected,
    },
    "complete": (
        train_present == train_expected
        and validation_present == validation_expected
    ),
}

print(json.dumps(report, indent=2))

In [ ]:
print({
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "cacheRoot": str(CACHE_ROOT),
    "trainCached": len(
        list((CACHE_ROOT / "train").glob("*.npy"))
    ),
    "validationCached": len(
        list((CACHE_ROOT / "validation").glob("*.npy"))
    ),
})

## Entrenamiento

Se selecciona el mejor checkpoint usando exclusivamente métricas de validación. El proceso usa early stopping y no ajusta nada con el internal test.


In [ ]:
# 8) Entrenar y evaluar sobre validation
summary = train_model(
    train_samples=train_samples,
    validation_samples=validation_samples,
    cache_root=CACHE_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    run_root=RUN_ROOT,
    manifest_hashes=manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# 9) Gate final y artefactos
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name
    for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]
if missing_outputs:
    raise RuntimeError(f"Faltan outputs: {missing_outputs}")

if summary["approved"] is not True:
    raise RuntimeError(
        "El entrenamiento requiere revisión. "
        "No abrir el internal test ni ajustar gates con él."
    )
if summary["status"] != "APPROVED_FOR_NOTEBOOK_64":
    raise RuntimeError("Estado final inesperado.")
if summary["governance"]["internalTestAccessed"] is not False:
    raise RuntimeError("Se declaró acceso indebido al internal test.")

print({
    "status": summary["status"],
    "bestEpoch": summary["bestEpoch"],
    "validationMetrics": summary["validationMetrics"],
    "checkpoint": summary["checkpoint"],
    "outputs": required_outputs,
    "internalTestSealedUntilNotebook64": True,
    "officialTestAccessed": False,
})


## Resultado esperado

Una ejecución aprobada termina con `APPROVED_FOR_NOTEBOOK_64`.

El Notebook 64 abrirá una sola vez `internal_test_manifest.csv`, evaluará el checkpoint congelado y exportará el modelo final. Si algún gate falla, se conserva la evidencia de validación y no se utiliza el internal test para ajustar el modelo.
